# 🔎 DefectSpotter — visual inspection without defect labels (end-to-end Colab)

**Project 3 · Manufacturing.** Factories rarely have many images of defects; most parts are good. DefectSpotter learns
**only from good parts**, flags anything unusual, shows **where** the anomaly is and turns it into a
**PASS / CHECK / REJECT** decision for the quality inspector at the end of the line.

This notebook runs the complete project, from EDA to the interactive Gradio dashboard and the Hugging Face deployment:

| Step | What happens | Course link |
|---|---|---|
| 1–3 | Clone code, configure, download MVTec AD (bottle, screw, carpet + metal nut) | data |
| 4–5 | Leakage-free splits and **EDA** (composition, acquisition statistics, defect sizes/locations) | Weeks 1–3 |
| 6–7 | **Synthetic defects**: procedural cut-paste / scratch / stain + **Stable-Diffusion inpainting** | Week 8 (GenAI) |
| 8 | **PatchCore-style** memory bank on frozen ResNet-18 features, greedy coreset, label-free tuning | Weeks 6–7 |
| 9 | **Convolutional VAE** (GenAI detector) trained on good parts | Week 8 |
| 10 | **Classical baseline**: ECC-aligned mean good image | Weeks 2–3 |
| 11 | Calibrated 0–100 score scale + **cost-optimal three-way thresholds** from synthetic defects only | decision theory |
| 12 | **Evaluation**: image/pixel AUROC, AUPRO, FRR, escape rate, cost table, synthetic-vs-real threshold, VAE vs PatchCore | evaluation |
| 13 | **Latency** per part and deployment footprint (ONNX ≈ 11 MB, banks ≪ 50 MB) | deployment |
| 14 | **Drift**: lighting/camera perturbations, camera-health gate, EWMA + PSI alarms | Lecture D |
| 15 | **Few-shot onboarding** of a new product (metal nut) | stretch goal |
| 16–19 | Tests, Gradio dashboard (public link), Hugging Face Space deployment, ZIP/Drive export | delivery |

> **Licence.** MVTec AD is CC BY-NC-SA 4.0 — research and teaching only, no commercial use.
> **Runtime.** Works on CPU (≈1–1.5 h for the full profile); a GPU runtime (Runtime → Change runtime type → T4) is ~5× faster
> and is recommended for the diffusion step. `PROFILE = "quick"` finishes in ~15 minutes.

## 1 · Get the code and install dependencies

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/hbasired/defectspotter.git"
REPO_BRANCHES = ["main", "claude/optimistic-mccarthy-q7e5mh"]   # first branch that contains the package wins
IN_COLAB = "google.colab" in sys.modules
ROOT = Path("/content/defectspotter" if IN_COLAB else "defectspotter").resolve()

if not (ROOT / "defectspotter" / "pipeline.py").exists():
    for branch in REPO_BRANCHES:
        subprocess.run(["rm", "-rf", str(ROOT)])
        result = subprocess.run(["git", "clone", "--depth", "1", "-b", branch, REPO_URL, str(ROOT)], capture_output=True, text=True)
        if result.returncode == 0 and (ROOT / "defectspotter" / "pipeline.py").exists():
            print(f"Cloned branch {branch}")
            break
    else:
        raise RuntimeError("Could not clone the DefectSpotter repository: " + result.stderr[-500:])
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
# Colab already ships CUDA PyTorch; install the rest (diffusers only when the diffusion step is enabled below).
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt"])
import torch, gradio, onnxruntime
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| gradio", gradio.__version__, "| onnxruntime", onnxruntime.__version__)

## 2 · Configure the run
Every number in the app and the report is produced by this run — nothing is pre-filled. Costs are *illustrative* and
editable: the economically right threshold depends on defect prevalence, the cost of an escape, the cost of scrapping a good
part and inspector time.

In [ ]:
from defectspotter.decision import CostModel
from defectspotter.pipeline import Pipeline, PipelineConfig

PROFILE = "full"            # "quick" (≈15 min smoke run) or "full" (research run)
CATEGORIES = ("bottle", "screw", "carpet")
FEW_SHOT_CATEGORY = "metal_nut"
RUN_DIFFUSION = True        # Stable-Diffusion inpainting of synthetic defects (GPU strongly recommended)
RUN_BACKBONE_STUDY = False  # ~25 min on CPU: compares MobileNetV3 / EfficientNet / ResNet-18 / WRN-50
USE_DRIVE = False           # keep caches + outputs in Google Drive (resumable)
DEPLOY_TO_HF = False        # needs a Colab secret HF_TOKEN with write access
HF_SPACE_ID = "bhkr0901/defectspotter"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

COSTS = CostModel(defect_prevalence=0.01, cost_escape=400.0, cost_false_reject=2.0, cost_review=0.30,
                  review_capacity=0.10, human_detection_rate=0.80, parts_per_day=20000)
DATA_DIR = Path("/content/mvtec_ad" if IN_COLAB else "data/mvtec_ad")
WORK_DIR = Path("/content/defectspotter_work")
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/DefectSpotter/work")

cfg = PipelineConfig(data_root=str(DATA_DIR), project_root=str(ROOT), work_dir=str(WORK_DIR), categories=CATEGORIES,
                     few_shot_category=FEW_SHOT_CATEGORY, device=DEVICE, costs=COSTS,
                     vae_epochs=120 if PROFILE == "full" else 10, bootstrap=1000 if PROFILE == "full" else 200)
if PROFILE == "quick":
    cfg.coreset_grid, cfg.blur_grid, cfg.topk_grid = (0.05,), (4.0,), (0.01,)
pipe = Pipeline(cfg)
print(cfg)

## 3 · Download MVTec AD (pinned Hugging Face mirror)

In [ ]:
from defectspotter.constants import MVTEC_HF_REPO, MVTEC_HF_REVISION
from defectspotter.data import download_mvtec, index_mvtec
download_mvtec(DATA_DIR, [*CATEGORIES, FEW_SHOT_CATEGORY])
print(f"Mirror {MVTEC_HF_REPO} @ {MVTEC_HF_REVISION[:8]}  ·  licence CC BY-NC-SA 4.0 (non-commercial)")
index_mvtec(DATA_DIR, [*CATEGORIES, FEW_SHOT_CATEGORY]).groupby(["category", "split", "label"]).size().unstack(fill_value=0)

## 4 · Leakage-free splits
| role | used for |
|---|---|
| `model_train_good` | fitting the memory bank, the VAE and the mean image (80% of train/good) |
| `calibration_good` | score scale, hyper-parameter search and threshold (20% of train/good, never seen by the models) |
| `oracle_calibration` | ~35% of the labelled test set: **only** to compute the "real-defect" threshold for comparison |
| `operating_test` | the rest of the test set: false-reject / escape / cost evaluation |
No real defect image is ever used to fit a detector or to pick the deployed threshold.

In [ ]:
pipe.stage_index()
import pandas as pd
splits = pd.read_csv(ROOT / "reports" / "splits.csv")
splits.groupby(["category", "role", "label"]).size().unstack(fill_value=0)

## 5 · Exploratory data analysis

In [ ]:
pipe.stage_eda()
from ui import charts
from ui.store import ReportStore
from ui.app import _eda_insights
from IPython.display import Image as IPImage, Markdown, display
store = ReportStore(ROOT)
comp, stats = store.csv("dataset_composition.csv"), store.csv("image_stats.csv")
charts.composition_chart(comp).show()
for c in CATEGORIES:
    charts.defect_types_chart(comp, c).show()
for signal in ("brightness", "contrast", "sharpness"):
    charts.acquisition_box(stats, signal).show()
charts.mask_area_chart(stats).show()
display(Markdown(_eda_insights(stats, comp)))
for c in [*CATEGORIES, FEW_SHOT_CATEGORY]:
    display(Markdown(f"**{c}** — good parts and one example per defect type (red = ground truth)"))
    display(IPImage(filename=str(ROOT / "reports" / "figures" / f"eda_montage_{c}.jpg"), width=760))

In [ ]:
# Where do defects occur? (sum of all ground-truth masks per product)
for c in CATEGORIES:
    display(Markdown(f"Defect location heat map · **{c}**"))
    display(IPImage(filename=str(ROOT / "reports" / "figures" / f"eda_defect_location_{c}.png"), width=220))

## 6 · Synthetic defects (procedural)
Held-out good parts are corrupted with **cut-paste** (CutPaste, Li et al. 2021), thin **scratches** and **stains**, each with an
exact mask. They let us tune and calibrate *without real defects*. They are not proof of real-defect coverage — §12 measures
how far the synthetic threshold lands from the one real labels would choose.

In [ ]:
pipe.stage_synthetic()
import matplotlib.pyplot as plt
gallery = store.synthetic_gallery()
fig, axes = plt.subplots(3, 6, figsize=(16, 8.5))
for ax, (path, caption) in zip(axes.ravel(), gallery):
    ax.imshow(plt.imread(path)); ax.set_title(caption, fontsize=9); ax.axis("off")
for ax in axes.ravel()[len(gallery):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 7 · Generative synthetic defects: Stable-Diffusion refinement (GenAI)
Plain *inpainting* fills a masked hole from its context, so on a good part it re-draws a good surface (we measured mean
changes of 2–6 grey levels: invisible). Instead each procedural defect is re-rendered with **SDEdit**: the base **SD-1.5**
model + **LCM-LoRA** (6 steps, strength 0.45) on a crop around the defect, with product-specific prompts ("a crack and a
chipped edge on the rim of a glass bottle", "a dark wine stain soaked into carpet fabric", ...). The defect keeps its
structure and gains realistic texture; only the masked region is pasted back. A label-free capability check (§11) decides
whether these images are reliable enough to influence the threshold.

In [ ]:
if RUN_DIFFUSION:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-diffusion.txt"])
    subprocess.check_call([sys.executable, "scripts/generate_diffusion_defects.py", "--data", str(DATA_DIR),
                           "--work-dir", str(WORK_DIR), "--per-category", "12" if DEVICE == "cuda" else "8"])
    gallery = store.diffusion_gallery()
    fig, axes = plt.subplots(2, 6, figsize=(16, 6))
    for ax, (path, caption) in zip(axes.ravel(), gallery):
        ax.imshow(plt.imread(path)); ax.set_title(caption, fontsize=8); ax.axis("off")
    for ax in axes.ravel()[len(gallery):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("Diffusion step skipped (RUN_DIFFUSION = False); the procedural synthetic set is still used.")

## 8 · PatchCore-style detector (Weeks 6–7)
* **Features**: frozen ImageNet ResNet-18, `layer2` + `layer3` (strides 8/16), 3×3 neighbourhood averaging → 384-d patch
  descriptors on a 28×28 grid. Exported once to ONNX (≈11 MB) and shared by all products.
* **Memory bank**: all good training patches compressed with **greedy k-center coreset** selection (keeps rare-but-normal
  patches that random subsampling drops), stored in float16.
* **Score**: distance of every test patch to its nearest normal patch → anomaly map (bilinear upsampling + Gaussian σ);
  image score = max (or top-k mean) of the patch distances.
* **Tuning without defects**: coreset ratio, smoothing and aggregation are chosen on held-out good + *synthetic* parts using
  image AUROC, pixel AUROC and the share of synthetic defects caught at ≤1% false rejects.

In [ ]:
pipe.stage_patchcore()
hpo = pd.read_csv(ROOT / "reports" / "hpo_patchcore.csv")
hpo.sort_values(["category", "objective"], ascending=[True, False]).groupby("category").head(4).round(4)

## 9 · Convolutional VAE trained on good parts (Week 8 · GenAI detector)
Encoder (4 strided conv blocks) → 1×1 squeeze → 128-d Gaussian latent → mirrored decoder; loss = L1 + 0.5·(1−SSIM) + β·KL.
Augmentation respects the product (no flips for screws — that would change thread handedness). The anomaly map is
0.5·|x − x̂| + 0.5·(1 − SSIM(x, x̂)); the image score is the top-1% mean. Exported to ONNX (≈9 MB per product).

In [ ]:
pipe.stage_vae()
charts.vae_history_chart(pd.read_csv(ROOT / "reports" / "vae_training_history.csv")).show()

## 10 · Classical baseline: difference from the mean good image after alignment (Weeks 2–3)
Images are aligned to a median good reference with OpenCV's ECC (translation / Euclidean / affine — chosen on synthetic
validation data), then compared with the per-pixel mean and standard deviation of aligned good parts.

In [ ]:
pipe.stage_mean()
display(pd.read_csv(ROOT / "reports" / "hpo_mean_baseline.csv").round(4))
for c in CATEGORIES:
    display(Markdown(f"Mean good image | per-pixel variability · **{c}**"))
    display(IPImage(filename=str(ROOT / "reports" / "figures" / f"baseline_mean_std_{c}.jpg"), width=460))

## 11 · Calibration and cost-optimal thresholds (no real defects)
* **Score scale** (0–100): 25 = median held-out good part, 75 = median synthetic defect.
* **Pixel scale**: 0 = median good pixel, 50 = 99.9th percentile of good pixels, 100 = typical synthetic-defect pixel.
* **Three-way rule**: PASS < t − b ≤ CHECK < t + b ≤ REJECT. `(t, b)` minimise the expected cost per part
  `(1−p)·FRR·C_fr + p·escape·C_esc + review·C_review` subject to inspector capacity (≤ 10% of parts in CHECK).
* **Detection-limit calibration:** every synthetic defect is replayed at 25% and 50% severity. Real screw thread damage is
  far subtler than full-strength synthetic defects; calibrating only on conspicuous defects set the screw threshold too high
  (35% escapes in the first run). Inspection specs define the faintest defect that must be caught — this sweep mirrors that.
* Policies stored per detector: `synthetic_cost` (deployed default, 25/50/100% severity), `synthetic_conspicuous_cost`
  (full-strength only, first version), `good_quantile` (1% FRR, needs no defects at all),
  `diffusion_cost`, `synthetic_plus_diffusion_cost` and `real_oracle_cost` (labelled defects — comparison only).
* The ensemble weight (PatchCore vs VAE) is also chosen on synthetic data.

In [ ]:
pipe.stage_severity()      # detection-limit set: every synthetic defect replayed at 25% and 50% severity
pipe.stage_diffusion()
pipe.stage_calibrate()
import json
rows = []
for c in CATEGORIES:
    cal = json.loads((ROOT / "artifacts" / c / "calibration.json").read_text())
    for m, info in cal["methods"].items():
        for name, pol in info["policies"].items():
            rows.append({"category": c, "method": m, "policy": name, **pol})
pd.DataFrame(rows).pivot_table(index=["category", "method"], columns="policy", values="threshold").round(1)

## 12 · Evaluation on held-out real defects

In [ ]:
pipe.stage_evaluate()
store = ReportStore(ROOT)   # reload the fresh reports
metrics = store.csv("metrics.csv")
display(metrics.round(4))
for metric, label in [("image_auroc", "Image AUROC"), ("pixel_auroc", "Pixel AUROC"), ("aupro_30", "AUPRO @ 30% FPR")]:
    charts.metric_bars(metrics, metric, label).show()

In [ ]:
# VAE vs feature method, per product: ROC curves, score scatter and decisions per defect type
for c in CATEGORIES:
    charts.roc_chart(store.json("roc_curves.json"), c).show()
    charts.method_scatter(store.json("scores.json"), c).show()
    charts.defect_type_bars(store.csv("per_defect_type.csv"), c, "patchcore").show()

In [ ]:
# Qualitative panels: median-scoring test part per defect type (input | ground truth | PatchCore | VAE | mean baseline)
for c in CATEGORIES:
    display(Markdown(f"**{c}**"))
    display(IPImage(filename=str(ROOT / "reports" / "figures" / f"qualitative_{c}.jpg"), width=900))

In [ ]:
# Operating metrics, cost table and the synthetic-vs-real threshold question
ops = store.csv("operating_metrics.csv")
display(ops[ops.policy.isin(["synthetic_cost", "real_oracle_cost", "good_quantile"])][
    ["category", "method", "policy", "threshold", "band", "false_reject_rate", "escape_rate", "review_rate",
     "reject_precision_at_prevalence", "expected_cost_per_1000"]].round(3))
thr = store.csv("threshold_comparison.csv")
display(thr.round(2))
for m in ("patchcore", "vae"):
    charts.threshold_dumbbell(thr, m).show()
from ui.story import cost_summary
cs = cost_summary(store)
charts.cost_comparison(store.json("cost_model.json"), cs["ours"]).show()
print("Cost model (per part):", json.dumps(store.json("cost_model.json"), indent=1))

## 13 · Latency per part and deployment footprint

In [ ]:
pipe.stage_latency()
store = ReportStore(ROOT)
display(store.csv("latency.csv").round(1))
charts.latency_bars(store.csv("latency.csv")).show()
charts.accuracy_latency(store.csv("metrics.csv"), store.csv("latency.csv")).show()

## 14 · Drift: lighting and camera changes (Lecture D)
A dimming lamp, defocus, a bumped camera, white-balance shifts, sensor noise or glare are applied to **good** parts at five
severities. We measure how often they become false rejects, how often the **camera-health gate** (brightness / contrast /
focus z-scores vs. qualified images) catches them, and whether real defects still get caught. A simulated 200-part stream
(qualified → lamp ageing → maintenance → camera bumped) is monitored with **EWMA control charts** and **PSI**.

In [ ]:
pipe.stage_drift()
store = ReportStore(ROOT)
stream, sens = store.csv("drift_stream.csv"), store.csv("drift_sensitivity.csv")
for c in CATEGORIES:
    pol = json.loads((ROOT / "artifacts" / c / "calibration.json").read_text())["methods"]["patchcore"]["policies"]["synthetic_cost"]
    charts.drift_stream(stream, c, pol["threshold"], pol["band"]).show()
    charts.psi_chart(stream, c, store.score_groups(c, "patchcore")["cal_good"]).show()
    charts.health_stream(stream, c, "brightness").show()
    charts.drift_heatmap(sens, c, "good_false_reject_rate", f"False-reject rate on good parts under drift · {c}").show()
    charts.drift_heatmap(sens, c, "health_flag_rate", f"Caught by the camera-health gate · {c}").show()
display(store.csv("drift_alarms.csv"))

## 15 · Few-shot onboarding of a new product (metal nut)

In [ ]:
pipe.stage_few_shot()
few = pd.read_csv(ROOT / "reports" / "few_shot.csv")
charts.few_shot_chart(few).show()
few.groupby("shots")[["image_auroc", "pixel_auroc", "false_reject_rate", "escape_rate"]].mean().round(3)

## 16 · Examples for the app, summary and (optional) backbone study

In [ ]:
pipe.stage_examples()
pipe.stage_summary()
if RUN_BACKBONE_STUDY:
    subprocess.check_call([sys.executable, "scripts/backbone_study.py", "--data", str(DATA_DIR), "--out", "reports/backbone_study.csv"])
store = ReportStore(ROOT)
charts.backbone_chart(store.csv("backbone_study.csv")).show()
charts.footprint_chart(store.csv("artifact_sizes.csv")).show()
print(json.dumps(store.json("summary.json"), indent=1)[:2500])

## 17 · Tests

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"])
print(subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True).stdout[-1500:])

## 18 · Launch the full Gradio dashboard
Opens the same app that is deployed to Hugging Face: project story, live inspection (upload / paste / webcam / ~1 fps
stream), batch + CSV, human review queue + audit log, threshold & cost lab, EDA, model evaluation, drift monitor, few-shot
onboarding and the `/inspect_part`, `/inspect_batch`, `/health` API. `share=True` prints a temporary public link
(valid while this runtime is connected).

In [ ]:
import importlib, ui.app
importlib.reload(ui.app)
demo = ui.app.build_app(ROOT)
demo.queue(default_concurrency_limit=4).launch(share=True, debug=False, **ui.app.launch_kwargs())

## 19 · Deploy to Hugging Face Spaces (free ZeroGPU tier)
Hugging Face rules (2026): free personal accounts with a verified e-mail and an account older than 30 days can host up to
**2 Gradio Spaces on ZeroGPU** for free; CPU-basic Gradio Spaces need PRO. DefectSpotter's inference runs on the Space's CPU
via ONNX Runtime, so visitors never consume GPU quota.

1. Create a **write** token at https://huggingface.co/settings/tokens.
2. Colab → 🔑 **Secrets** → add `HF_TOKEN` and enable notebook access. Never paste the token into a cell.
3. Set `DEPLOY_TO_HF = True` in §2 and run this cell.

In [ ]:
if DEPLOY_TO_HF:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    subprocess.check_call([sys.executable, "scripts/deploy_hf.py", "--space", HF_SPACE_ID, "--hardware", "zero-a10g"])
    os.environ.pop("HF_TOKEN", None)
else:
    print("Deployment skipped (DEPLOY_TO_HF = False). Dry run of the bundle:")
    subprocess.run([sys.executable, "scripts/deploy_hf.py", "--space", HF_SPACE_ID, "--dry-run"])

## 20 · Package everything (ZIP + optional Drive copy)

In [ ]:
import shutil
archive = shutil.make_archive("/content/DefectSpotter_results" if IN_COLAB else "DefectSpotter_results", "zip", ROOT,
                              base_dir=".")
print("Created", archive)
if USE_DRIVE and IN_COLAB:
    shutil.copy(archive, "/content/drive/MyDrive/DefectSpotter/")
if IN_COLAB:
    from google.colab import files
    files.download(archive)

## Completion checklist
- [x] MVTec AD bottle / screw / carpet (+ metal nut) downloaded from a pinned mirror, licence surfaced
- [x] Leakage-free splits; EDA tables, charts, montages, defect sizes and locations
- [x] Procedural + Stable-Diffusion synthetic defects with exact masks
- [x] PatchCore-style memory bank (greedy coreset, ONNX extractor), convolutional VAE, aligned mean baseline, ensemble
- [x] Label-free tuning and cost-optimal PASS/CHECK/REJECT thresholds; synthetic vs real-defect threshold gap
- [x] Image AUROC, pixel AUROC, AUPRO, FRR, escape rate, cost table, latency, footprint
- [x] Drift stress test, camera-health gate, EWMA + PSI monitoring; few-shot onboarding
- [x] Gradio dashboard + API; Hugging Face ZeroGPU deployment; ZIP export

**Responsible use.** CHECK decisions always go to a trained inspector; heat maps show visual deviation, not root cause; drift
alarms trigger maintenance and re-qualification, never silent retraining; MVTec AD is non-commercial.